# Semantic Chunking for RAG via BiLSTM

Trains a semantic boundary detector on Wikipedia section pseudo-labels and evaluates retrieval on Natural Questions.

Requires a GPU runtime and a project directory configured through `RAG_PROJECT_DIR`. Data, weights, and results use the configured storage root. Experiment settings are in `config.py`.


## 0. Setup

In [ ]:
# torch is pre-installed on Colab; install the rest.
!pip -q install datasets sentence-transformers faiss-cpu mwparserfromhell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
# Set RAG_PROJECT_DIR to the project root before running this cell.
PROJECT_DIR = os.environ.get('RAG_PROJECT_DIR')
if not PROJECT_DIR:
    raise RuntimeError('Set RAG_PROJECT_DIR to the project directory before running this notebook.')
os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
sys.path.insert(0, PROJECT_DIR)
# Stand in the project root so later `!python scripts/...` cells find scripts/.
os.chdir(PROJECT_DIR)

import config as C
C.ensure_dirs()
print(C.summary())

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())

## Smoke test

Runs five phases with about 30 Wikipedia articles, one training epoch, and 10 NQ documents in a separate `*_smoke` directory. This checks the small pipeline path, not retrieval quality. The original configuration is restored afterward.


In [ ]:
from rag_chunk import smoke
_ = smoke.run_smoke()

## Phase 1 - Data preparation (Wikipedia -> sentences + boundary labels)

Picks article titles by streaming the HF dump, fetches **raw wikitext** via the
MediaWiki API (batched, cached, resumable), parses real `== Section ==` structure
with `mwparserfromhell`, and writes `train/val/test.jsonl`.

In [ ]:
from rag_chunk import wiki_data
stats = wiki_data.prepare_dataset(C.N_WIKI_ARTICLES)
stats

## Phase 2 - Offline embedding (all-MiniLM-L6-v2)

Caches one `(n_sentences, 384)` array per article to Drive. Resumable.

In [ ]:
from rag_chunk import embedding
embedding.embed_offline()

## Phase 3 - Train the BiLSTM boundary detector

Weighted BCE (`pos_weight = #neg/#pos`), one article per step, early stopping on
validation loss. Best weights are saved to Drive.

In [ ]:
from rag_chunk import training
train_stats = training.train_model()
train_stats['test_boundary_f1']

## Phase 4 - Build RAG indices on Natural Questions

Streams NQ `validation`, chunks each document two ways (BiLSTM vs fixed 5-sentence),
embeds the chunks, and builds two FAISS `IndexFlatIP` indices.

In [ ]:
from rag_chunk import retrieval, training
model = training.load_model()
built = retrieval.build_indexes(model)
print('bilstm chunks:', len(built['bilstm'].chunk_texts),
      '| fixed chunks:', len(built['fixed'].chunk_texts),
      '| questions:', len(built['questions']))

## Phase 5 - Evaluate (Recall@k + Boundary F1)

Prints the comparison table and writes `results/recall_comparison.csv` and `.png`.

In [ ]:
from rag_chunk import evaluation, training
model = training.load_model()
results = evaluation.evaluate_all(model)
results

In [ ]:
from IPython.display import Image
Image(filename=str(C.RESULTS_FIGURE))

## Phase 6 - Chunking sweep optimizer

Sweeps **fixed-size** and learned **target-size** chunking across chunk sizes and
overlaps, scores each on NQ doc-constrained Recall@k, and exports optimizer
artifacts to `artifacts/results/latest/`: `sweep_results.csv`, `best_config.json`,
`fair_comparison_table.csv`, `recall_vs_chunk_size.png`.

This stage does **not** assume learned chunking always wins — it controls for chunk
size and overlap, then measures which strategy retrieves answer-bearing chunks
better. Per-config FAISS indices are built in memory and discarded (nothing is
cached under `nq/` unless `--save-sweep-index` is enabled).

In [ ]:
from rag_chunk import sweep, training
model = training.load_model()
# Fast grid first (quick=True). For the full grid use quick=False, or run
# `python scripts/6_sweep_chunking.py`. run_sweep prints the ranked table + best config.
rows = sweep.run_sweep(model, quick=True)
print(f"\n{len(rows)} configs swept -> artifacts/results/latest/")

In [ ]:
from IPython.display import Image
Image(filename=str(C.RESULTS_LATEST_DIR / C.RECALL_PLOT_PNG))

## Phase 7 - Train Transformer boundary model (Stage 2)

Trains a 2-layer Transformer boundary detector as a **second** learned
chunking model. It reuses the **same** cached MiniLM sentence embeddings and
Wikipedia section labels from Phases 1-3 — **no need to rerun Phase 1/2**.
Best weights are saved to `models/transformer_best.pt`; the BiLSTM weights
(`models/bilstm_best.pt`) are left untouched.

In [ ]:
from rag_chunk import training
tf_stats = training.train_model(model_type="transformer")
tf_stats['test_boundary_f1']

## Phase 8 - Compare Fixed vs BiLSTM vs Transformer (Stage 2)

Runs the Phase 6 sweep with the Transformer added as a third method, under the
same MiniLM embeddings, the same cached NQ docs/questions, the same Recall@k
metric and the same target-size/overlap policy — so only the boundary model
differs. Writes `sweep_results.csv`, `best_config.json`,
`fair_comparison_table.csv`, `recall_vs_chunk_size.png` and
`model_comparison.png` to `artifacts/results/latest/`.

`quick=True` uses `{8, 10, 12}` for a preview. Stage 3 requires the full-grid Stage 2 archive `{6, 8, 10, 12, 15}`.


In [ ]:
from rag_chunk import sweep, training
bilstm = training.load_model("bilstm")
transformer = training.load_model("transformer")
# quick=True for a fast grid; drop it (or run scripts/8_sweep_with_transformer.py) for the full grid.
rows = sweep.run_sweep(bilstm, transformer_model=transformer, quick=True)
print(f"\n{len(rows)} configs swept (fixed + bilstm + transformer) -> artifacts/results/latest/")

In [ ]:
from IPython.display import Image
Image(filename=str(C.RESULTS_LATEST_DIR / C.RECALL_PLOT_PNG))

## Stage 3: BGE retrieval embedding ablation

Keeps the MiniLM boundary embeddings and BiLSTM/Transformer weights while replacing retrieval embeddings with BGE.

The Stage 2 archive must use the full grid `{6, 8, 10, 12, 15}`. The quick Phase 8 grid `{8, 10, 12}` omits configurations needed for the matched comparison. The following cell runs and archives the full Stage 2 grid before Stage 3 overwrites `artifacts/results/latest/`.


In [ ]:
# Full Stage 2 grid required for the matched Stage 3 comparison.
!python scripts/8_sweep_with_transformer.py
!python scripts/save_stage_results.py --stage stage2

In [ ]:
# Full Stage 3 BGE retrieval sweep.
# If Colab is slow, switch to:
# !python scripts/9_sweep_bge_retrieval.py --retrieval-model BAAI/bge-small-en-v1.5
!python scripts/9_sweep_bge_retrieval.py


In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.SWEEP_RESULTS_CSV,
    C.BEST_CONFIG_JSON,
    C.FAIR_TABLE_CSV,
    'stage2_vs_stage3_matched.csv',
    C.RECALL_PLOT_PNG,
    C.MODEL_PLOT_PNG,
    'stage3_bge_retrieval_summary.md',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(Image(filename=str(latest / 'recall_vs_chunk_size.png')))
display(Image(filename=str(latest / 'model_comparison.png')))

In [ ]:
import pandas as pd
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / 'sweep_results.csv'))
display(pd.read_csv(latest / 'fair_comparison_table.csv'))

## Stage 4 - Hybrid retrieval ablation (BM25 + BGE + RRF)

Same chunks, three retrievers per chunking config: **`bge`** (dense, exactly the Stage 3 path), **`bm25`** (lexical Okapi BM25 — pure numpy, no new install), and **`rrf`** (Reciprocal Rank Fusion of the two rankings, k=60, depth 50). Dataset size, chunking grids, boundary models/weights and the BGE model are all unchanged from Stage 3.

Run order:

1. **Archive Stage 3** (copy-only). Stage 4 refuses to start without `artifacts/results/stage3/final/`, and it only clears files from `latest/` that have a byte-identical archived copy — so nothing unarchived is ever deleted and `stage3/final` is never touched.
2. **Run the sweep.** It re-runs the BGE arm and compares it row-by-row against the archived Stage 3 baseline (`stage3_vs_stage4_bge_check.csv`): every `delta_n_chunks` must be 0 and every recall delta 0.0000 — the console prints **"check OK"** when the baseline is reproduced exactly.
3. **Only after the check passes**, archive Stage 4 to `artifacts/results/stage4/final/`.

In [ ]:
!python scripts/save_stage_results.py --stage stage3

In [ ]:
!python scripts/10_sweep_hybrid_retrieval.py

In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.HYBRID_SWEEP_CSV,
    C.HYBRID_BEST_JSON,
    C.HYBRID_MATCHED_CSV,
    'stage3_vs_stage4_bge_check.csv',
    C.HYBRID_SCATTER_PNG,
    C.HYBRID_RETRIEVER_PLOT_PNG,
    'stage4_hybrid_retrieval_summary.md',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(Image(filename=str(latest / C.HYBRID_SCATTER_PNG)))
display(Image(filename=str(latest / C.HYBRID_RETRIEVER_PLOT_PNG)))

In [ ]:
import pandas as pd
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / C.HYBRID_MATCHED_CSV))
display(pd.read_csv(latest / 'stage3_vs_stage4_bge_check.csv'))

In [ ]:
# Requires a successful Stage 3 baseline comparison.
!python scripts/save_stage_results.py --stage stage4

## Stage 5 - Cross-encoder reranking (BGE top-k + bge-reranker-base)

Same chunks, three arms per chunking config: **`bge`** (dense-only, exactly the Stage 3/4 path), **`rerank20`** (BGE top-20 candidates reordered by the pretrained cross-encoder `BAAI/bge-reranker-base`), and **`rerank50`** (the same with a top-50 pool). Dataset size, chunking grids, boundary models/weights and the BGE dense retriever are all unchanged; the reranker is off-the-shelf — **no training or fine-tuning**. Goal metric: **Recall@1 / Recall@3** (Recall@5 is already near its ceiling).

Every row also reports `pool_recall@{20,50}` — whether the answer chunk was in the BGE candidate pool at all. Reranking can only promote chunks the pool already contains, so read any improvement against that ceiling.

Run order:

1. **Run the sweep.** No archive step needed first — Stage 4 step 3 above already archived `latest/` to `stage4/final/`, and the script refuses to delete anything from `latest/` without a byte-identical archived copy (`stage3/final` and `stage4/final` are only ever read). It re-runs the BGE arm and compares it row-by-row against the archived Stage 3 baseline (`stage3_vs_stage5_bge_check.csv`) — the console prints **"check OK"** when the baseline is reproduced exactly.
2. **Only after the check passes**, archive Stage 5 to `artifacts/results/stage5/final/`.

In [ ]:
# Requires stage3/final and an archived results/latest directory.
!python scripts/11_sweep_reranker.py

In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.RERANK_SWEEP_CSV,
    C.RERANK_BEST_JSON,
    C.RERANK_MATCHED_CSV,
    'stage3_vs_stage5_bge_check.csv',
    C.RERANK_SCATTER_PNG,
    C.RERANK_COMPARISON_PNG,
    'stage5_reranker_summary.md',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(Image(filename=str(latest / C.RERANK_SCATTER_PNG)))
display(Image(filename=str(latest / C.RERANK_COMPARISON_PNG)))

In [ ]:
import pandas as pd
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / C.RERANK_MATCHED_CSV))
display(pd.read_csv(latest / 'stage3_vs_stage5_bge_check.csv'))

In [ ]:
# Requires a successful Stage 3 baseline comparison.
!python scripts/save_stage_results.py --stage stage5

## Stage 6 - Larger-scale robustness evaluation (~1000 docs)

Same dataset source, same chunking grids, same boundary models/weights, same BGE retriever, same off-the-shelf reranker (**top-20 only** — no rerank50, no fine-tuning). The only change: corpus scale **200 → ~1000 docs/questions**. Every stage so far ran at n=203 questions where 1 SE ≈ 0.034; at n≈1000 the SE halves, to reassess the Stage 1-5 findings — the actual loaded doc/question counts are printed and reported (the NQ stream stops at the N-th usable document, so they are not exactly 1000).

Arms: **`bge`** on the full 30-config grid; **`rerank20`** only on 5 selected configs (fixed 6/0, fixed 15/0, fixed 15/1, bilstm 15/0, transformer 15/0) — the small-chunk config plus the size-15 sweet spot, used by the direction checks below.

Run order:

1. **Check mode first.** Re-runs everything at N=200 on the same cached corpus and must reproduce the archived Stage 5 rows **exactly** (`stage6_check_vs_stage5.csv`) — wait for **"check OK"**. This checks agreement with the archived Stage 3/5 results on the cached corpus.
2. **The large run.** The bigger corpus caches under a separate `nq/large_n1000/` folder, so the 200-doc cache is untouched. The eval set changes, so no exact-delta check is possible — instead `stage6_direction_check.csv` re-tests the four Stage 1-5 direction claims (size > method; BGE strong; rerank20 helps mainly small chunks; no clear gain at the size-15 sweet spot) with explicit rules and observed values.
3. **Archive** to `artifacts/results/stage6/final/` after reviewing the direction checks. Stage 3/4/5 finals are only ever read.

In [ ]:
# Requires stage5/final; compares all rows with the archived Stage 5 results.
!python scripts/12_large_eval.py --check

In [ ]:
# Resume: latest/stage6_checkpoint_large.jsonl. --fresh starts a new run.
!python scripts/12_large_eval.py

In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.STAGE6_RESULTS_CSV,
    C.STAGE6_MATCHED_CSV,
    C.STAGE6_DIRECTION_CSV,
    C.STAGE6_SUMMARY_MD,
    'stage6_check_vs_stage5.csv',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
import pandas as pd
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / C.STAGE6_MATCHED_CSV))
display(pd.read_csv(latest / C.STAGE6_DIRECTION_CSV))
display(pd.read_csv(latest / 'stage6_check_vs_stage5.csv'))

In [ ]:
!python scripts/13_stage6_plots.py

from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
display(Image(filename=str(latest / C.STAGE6_SIZE_PLOT_PNG)))
display(Image(filename=str(latest / C.STAGE6_DELTA_PLOT_PNG)))

In [ ]:
# Requires completed check-mode, large evaluation, and figures.
!python scripts/save_stage_results.py --stage stage6

## Stage 7 - Cross-dataset robustness check (TriviaQA rc.wikipedia)

Same chunking grids, same boundary models/weights, same BGE retriever — the **only** change is the QA dataset: NQ → TriviaQA `rc.wikipedia`, whose full Wikipedia entity pages are bundled in the dataset (no fetching). **bge arm only** (no BM25/RRF, no reranking, no fine-tuning). Gold documents = the question's entity pages whose text contains the answer string (distant supervision — weaker than NQ's annotated gold; every output says so). See `docs/stage7_cross_dataset.md`.

Run order:

1. **Check mode first.** Re-runs the 30-config bge-only sweep on the cached NQ 200-doc corpus; every row must reproduce the archived Stage 3 rows **exactly** (`stage7_check_vs_stage3.csv`) — wait for **"check OK"**. This checks single-gold behaviour after the multi-gold metric extension.
2. **The TriviaQA run.** Streams ~300 kept questions (the loader prints filter statistics and aborts if the corpus would make the chunk-size sweep degenerate). No exact-delta check is possible across datasets — `stage7_direction_check.csv` re-tests the size-vs-method claims with explicit rules instead.
3. **Archive** to `artifacts/results/stage7/final/` after reviewing the direction checks. Stage 1-6 finals are only ever read.

In [ ]:
# Requires stage3/final; compares all 30 BGE configurations.
!python scripts/15_cross_dataset_eval.py --check

In [ ]:
# Resume: latest/stage7_checkpoint_trivia.jsonl. --fresh starts a new run.
!python scripts/15_cross_dataset_eval.py

In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.STAGE7_RESULTS_CSV,
    C.STAGE7_MATCHED_CSV,
    C.STAGE7_DIRECTION_CSV,
    C.STAGE7_SUMMARY_MD,
    C.STAGE7_SCATTER_PNG,
    'stage7_check_vs_stage3.csv',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / C.STAGE7_MATCHED_CSV))
display(pd.read_csv(latest / C.STAGE7_DIRECTION_CSV))
display(pd.read_csv(latest / 'stage7_check_vs_stage3.csv'))
display(Image(filename=str(latest / C.STAGE7_SCATTER_PNG)))

In [ ]:
# Requires completed check-mode and TriviaQA evaluation.
!python scripts/save_stage_results.py --stage stage7

## Stage 8 - Fine-tune the cross-encoder reranker (Route C)

Trigger (Stage 6): at the size-15 sweet spot `pool_recall@20` ≈ 0.96 but R@1 ≈ 0.63 and the **off-the-shelf** reranker adds ~0 — ranking, not pool recall, is the remaining bottleneck. Stage 8 fine-tunes `BAAI/bge-reranker-base` on the NQ **train** split (every eval bench uses the validation split, so train/eval stay disjoint). See `docs/stage8_reranker_finetune.md`.

Run order (**two sessions, with a go/no-go decision in between**):

1. **Build data** — stream the train split: training corpus (2000 docs) + dev bench (the next 400 docs), mine (1 positive + 7 hard negatives) groups from each question's BGE top-20 pool at the deployment chunking (fixed 15/0).
2. **Train** — listwise cross-entropy, fp16; saves each epoch to `models/bge_reranker_ft/`.
3. **Go/no-go on the dev bench** — dev ΔR@1 (ft − off-the-shelf) at fixed 15/0: ≥ +0.02 → **GO**; ≤ 0 → **NO-GO, STOP** (archive the negative result); in between → at most one retry. Model selection uses the dev bench; final reporting uses the separate evaluation bench.
4. **Final eval (ONLY if GO)** — the Stage 6 bench (1032 questions, 5 configs), three arms sharing one identical top-20 pool. Built-in check: the `bge` + `rerank20` rows must reproduce `stage6/final` **exactly** before the `rerank20_ft` rows mean anything.
5. **Archive** to `artifacts/results/stage8/final/`.

In [ ]:
!python scripts/16_build_rerank_train_data.py

In [ ]:
# Resume after epoch 1: --init-model "$RAG_DATA_ROOT/models/bge_reranker_ft/epoch1" --epochs 1
!python scripts/17_train_reranker.py

In [ ]:
# GO permits final evaluation; NO-GO archives the negative result.
# GRAY-ZONE permits at most one retry under the experiment protocol.
!python scripts/18_eval_reranker_ft.py --dev

In [ ]:
# Requires GO and exact reproduction of the archived Stage 6 baseline.
!python scripts/18_eval_reranker_ft.py

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

for name in [C.STAGE8_DEV_RESULTS_CSV, C.STAGE8_RESULTS_CSV,
             C.STAGE8_MATCHED_CSV, C.STAGE8_CHECK_CSV]:
    p = latest / name
    if p.exists():
        print(name)
        display(pd.read_csv(p))
if (latest / C.STAGE8_DELTA_PNG).exists():
    display(Image(filename=str(latest / C.STAGE8_DELTA_PNG)))

In [ ]:
# Archive dev results after NO-GO, or final results after GO and baseline checks.
!python scripts/save_stage_results.py --stage stage8

## Route D - Interactive demo (Gradio)

One question, two chunking strategies side by side (**fixed 15/0** vs **BiLSTM t15/0**) on the Stage 6 bench (1000 docs / 1032 questions), ranked by one of three arms sharing the same BGE top-20 pool: `bge` / `rerank20` (off-the-shelf) / `rerank20_ft` (the Stage 8 fine-tuned model). Bench questions highlight the answer, badge gold-document chunks, and show each chunk's dense-rank movement after reranking.

- Runs **no experiments** and writes **nothing** under `results/` — safe next to the archives.
- First run builds + caches two demo FAISS indices under `data/nq/large_n1000/indices/demo/` (embeds ~40k chunks); later runs load them from that cache.
- Click the public `*.gradio.live` link printed below; interrupt the cell to stop the server.

In [ ]:
# Route D: interactive demo (safe: read-only over the cached bench).
%pip install -q gradio
!python scripts/19_demo.py --share